In [1]:
import pandas as pd
import numpy as np
import random
from sklearn.preprocessing import MinMaxScaler

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

In [2]:
# STEP 1: LOAD NHTSA FILE 
# File has 49 columns but no header row, tab-separated
# We only load the 7 columns we need to keep memory low

col_names = [f'col_{i}' for i in range(49)]  # create 49 generic column names

df_nhtsa = pd.read_csv(
    'FLAT_CMPL.txt',
    sep='\t',                # tab separated
    encoding='latin-1',      # NHTSA files need this encoding
    header=None,             # no header row in the file
    names=col_names,
    usecols=[1, 2, 3, 4, 5, 11, 15],  # only load the columns we need
    low_memory=False
)

# Rename to meaningful names based on what we saw in the data
df_nhtsa.columns = ['complaint_id', 'manufacturer', 'make', 'model', 'year', 'component', 'date_complaint']

# Convert date column and year to proper formats
df_nhtsa['date_complaint'] = pd.to_datetime(df_nhtsa['date_complaint'], format='%Y%m%d', errors='coerce')
df_nhtsa['year'] = df_nhtsa['year'].astype(str)

print(df_nhtsa.shape)
df_nhtsa.head()

(2181155, 7)


,complaint_id,manufacturer,make,model,year,component,date_complaint
0,958241,"Volvo Car USA, LLC",VOLVO,760,1987.0,ENGINE AND ENGINE COOLING:COOLING SYSTEM:RADIA...,1995-01-03
1,958130,Ford Motor Company,FORD,THUNDERBIRD,1992.0,"FUEL SYSTEM, GASOLINE:DELIVERY",1995-01-03
2,958132,"Kia America, Inc.",KIA,SEPHIA,1994.0,POWER TRAIN:AUTOMATIC TRANSMISSION,1995-01-03
3,958133,"Chrysler (FCA US, LLC)",DODGE,600,1987.0,"FUEL SYSTEM, GASOLINE:STORAGE:TANK ASSEMBLY",1995-01-03
4,958137,"Chrysler (FCA US, LLC)",DODGE,CARAVAN,1991.0,SEATS,1995-01-03


In [4]:
# STEP 2: FILTER AND EXTRACT COMPONENT CATEGORY 
# Keep only complaints from 2015 onwards — older data not relevant
df_nhtsa = df_nhtsa[df_nhtsa['date_complaint'] >= '2015-01-01']

# Component column has format like "ENGINE AND ENGINE COOLING:COOLING SYSTEM:RADIATOR"
# We extract only the first part (top-level category) before the colon
df_nhtsa['component_category'] = df_nhtsa['component'].str.split(':').str[0].str.strip()

print(df_nhtsa.shape)
print(df_nhtsa['component_category'].value_counts().head(15))

(1037806, 8)
component_category
ENGINE                                136136
ELECTRICAL SYSTEM                     129730
AIR BAGS                               97146
POWER TRAIN                            97093
UNKNOWN OR OTHER                       94468
STEERING                               73326
SERVICE BRAKES                         59776
FUEL/PROPULSION SYSTEM                 48549
STRUCTURE                              44848
VEHICLE SPEED CONTROL                  29865
SUSPENSION                             29788
EXTERIOR LIGHTING                      27666
FORWARD COLLISION AVOIDANCE            24810
VISIBILITY/WIPER                       20128
ELECTRONIC STABILITY CONTROL (ESC)     15667
Name: count, dtype: int64


In [5]:
# STEP 3: LOAD DATACO SUPPLY CHAIN FILE 
# This file has delivery/logistics data — 180K rows, 53 columns
# latin-1 encoding needed here too

df_raw = pd.read_csv('DataCoSupplyChainDataset.csv', encoding='latin-1', low_memory=False)

# Keep only the columns relevant to our analysis
keep_cols = [
    'Days for shipping (real)', 'Days for shipment (scheduled)',
    'Delivery Status', 'Late_delivery_risk', 'Category Name',
    'Customer Country', 'Market', 'Order Country',
    'order date (DateOrders)', 'Order Id', 'Order Item Quantity',
    'Order Item Total', 'Order Profit Per Order', 'Order Region',
    'Order Status', 'shipping date (DateOrders)', 'Shipping Mode'
]

df_dataco = df_raw[keep_cols].copy()

# Rename to cleaner names
df_dataco.columns = [
    'days_shipping_real', 'days_shipping_scheduled', 'delivery_status',
    'late_delivery_risk', 'category', 'customer_country', 'market',
    'order_country', 'order_date', 'order_id', 'quantity', 'order_value',
    'profit', 'region', 'order_status', 'ship_date', 'shipping_mode'
]

# Parse date columns
df_dataco['order_date'] = pd.to_datetime(df_dataco['order_date'])
df_dataco['ship_date']  = pd.to_datetime(df_dataco['ship_date'])

# Calculate delay — positive means late, negative means early
df_dataco['delay_days'] = df_dataco['days_shipping_real'] - df_dataco['days_shipping_scheduled']
df_dataco['is_delayed'] = df_dataco['delay_days'] > 0  # True/False flag

print(df_dataco.shape)
print(df_dataco['delivery_status'].value_counts())

(180519, 19)
delivery_status
Late delivery        98977
Advance shipping     41592
Shipping on time     32196
Shipping canceled     7754
Name: count, dtype: int64


In [6]:
# STEP 4: MAP RETAIL CATEGORIES → AUTOMOBILE PART CATEGORIES
# DataCo is a general retail dataset. We map its product categories to
# auto component categories so it aligns with NHTSA complaint data.

category_map = {
    'Cleats'               : 'ENGINE',
    "Men's Footwear"       : 'ELECTRICAL SYSTEM',
    "Women's Apparel"      : 'SERVICE BRAKES',
    'Indoor/Outdoor Games' : 'POWER TRAIN',
    'Fishing'              : 'STEERING',
    'Water Sports'         : 'SUSPENSION',
    'Camping & Hiking'     : 'FUEL/PROPULSION SYSTEM',
    'Cardio Equipment'     : 'AIR BAGS',
    'Shop By Sport'        : 'STRUCTURE',
    'Electronics'          : 'VEHICLE SPEED CONTROL'
}

df_dataco['part_category'] = df_dataco['category'].map(category_map)

# Drop rows that didn't map to any auto category (the 'OTHER' ones)
df_dataco = df_dataco[df_dataco['part_category'].notna()].copy()

print(df_dataco.shape)
print(df_dataco['part_category'].value_counts())

(160351, 20)
part_category
ENGINE                    24551
ELECTRICAL SYSTEM         22246
SERVICE BRAKES            21035
POWER TRAIN               19298
STEERING                  17325
SUSPENSION                15540
FUEL/PROPULSION SYSTEM    13729
AIR BAGS                  12487
STRUCTURE                 10984
VEHICLE SPEED CONTROL      3156
Name: count, dtype: int64


In [7]:
#STEP 5: FIX COUNTRY NAME ENCODING 
# Country names were in Spanish with broken encoding (e.g. Mxico, Espaa)
# First fix the encoding, then translate Spanish → English

df_dataco['order_country'] = df_dataco['order_country'].str.encode('latin-1').str.decode('utf-8', errors='ignore')
df_dataco['region']        = df_dataco['region'].str.encode('latin-1').str.decode('utf-8', errors='ignore')
df_dataco['market']        = df_dataco['market'].str.encode('latin-1').str.decode('utf-8', errors='ignore')

country_translation = {
    'Estados Unidos': 'USA', 'Mxico': 'Mexico', 'México': 'Mexico',
    'Francia': 'France', 'Alemania': 'Germany', 'Brasil': 'Brazil',
    'Australia': 'Australia', 'Reino Unido': 'United Kingdom',
    'China': 'China', 'Italia': 'Italy', 'India': 'India',
    'El Salvador': 'El Salvador', 'Repblica Dominicana': 'Dominican Republic',
    'República Dominicana': 'Dominican Republic', 'Espaa': 'Spain',
    'España': 'Spain', 'Honduras': 'Honduras', 'Cuba': 'Cuba',
    'Indonesia': 'Indonesia', 'Turqua': 'Turkey', 'Turquía': 'Turkey',
    'Nicaragua': 'Nicaragua', 'Guatemala': 'Guatemala', 'Nigeria': 'Nigeria',
    'Argentina': 'Argentina', 'Colombia': 'Colombia', 'Venezuela': 'Venezuela',
    'Per': 'Peru', 'Perú': 'Peru', 'Chile': 'Chile', 'Ecuador': 'Ecuador',
    'Filipinas': 'Philippines', 'Pakistn': 'Pakistan', 'Pakistán': 'Pakistan',
    'Irn': 'Iran', 'Irán': 'Iran', 'Marruecos': 'Morocco',
    'Sudfrica': 'South Africa', 'Sudáfrica': 'South Africa', 'SudAfrica': 'South Africa',
    'Argelia': 'Algeria', 'Tnez': 'Tunisia', 'Túnez': 'Tunisia',
    'Japn': 'Japan', 'Japón': 'Japan', 'Corea del Sur': 'South Korea',
    'Tailandia': 'Thailand', 'Vietnam': 'Vietnam', 'Myanmar': 'Myanmar',
    'Banglads': 'Bangladesh', 'Bangladés': 'Bangladesh',
    'Panam': 'Panama', 'Pases Bajos': 'Netherlands', 'Nueva Zelanda': 'New Zealand',
    'Austria': 'Austria', 'Egipto': 'Egypt', 'Rusia': 'Russia',
    'Repblica Democrtica del Congo': 'DR Congo', 'República Democrática del Congo': 'DR Congo',
    'Irak': 'Iraq', 'Ucrania': 'Ukraine', 'Canada': 'Canada',
    'Suecia': 'Sweden', 'Arabia Saud': 'Saudi Arabia', 'Arabia Saudí': 'Saudi Arabia',
    'Polonia': 'Poland', 'Blgica': 'Belgium', 'Bélgica': 'Belgium',
    'Hait': 'Haiti', 'Rumania': 'Romania', 'Malasia': 'Malaysia',
    'Irlanda': 'Ireland', 'Tanzania': 'Tanzania'
}

df_dataco['order_country'] = df_dataco['order_country'].replace(country_translation)

print(f"Unique countries: {df_dataco['order_country'].nunique()}")
print(df_dataco['order_country'].value_counts().head(10))

Unique countries: 164
order_country
USA               23226
Mexico            12270
France            11563
Germany            8356
Brazil             7391
Australia          6632
United Kingdom     6376
China              4389
Italy              4330
India              3627
Name: count, dtype: int64


In [8]:
# STEP 6: GENERATE SUPPLIER MASTER TABLE 
# No public dataset gives a clean auto supplier list
# We generate 300 realistic suppliers using real industry names
# random.seed(42) and np.random.seed(42) ensure same results every time you run

random.seed(42)
np.random.seed(42)

manufacturers = [
    'Bosch', 'Denso', 'Magna International', 'Continental AG', 'ZF Friedrichshafen',
    'Aisin', 'Hyundai Mobis', 'Lear Corporation', 'Valeo', 'BorgWarner',
    'Motherson Sumi', 'Minda Industries', 'Bharat Forge', 'Sundram Fasteners',
    'Endurance Technologies', 'Schaeffler India', 'Tata AutoComp', 'Lumax Industries',
    'Samvardhana Motherson', 'Gabriel India', 'Subros', 'Uno Minda', 'WABCO India',
    'Aptiv', 'Faurecia', 'Tenneco', 'Delphi Technologies', 'Sensata Technologies'
]

countries       = ['India', 'Germany', 'Japan', 'USA', 'South Korea', 'China', 'Thailand', 'Mexico']
country_weights = [0.30, 0.15, 0.15, 0.10, 0.10, 0.10, 0.05, 0.05]  # India weighted heaviest — realistic

part_categories = [
    'ENGINE', 'ELECTRICAL SYSTEM', 'SERVICE BRAKES', 'POWER TRAIN',
    'STEERING', 'SUSPENSION', 'FUEL/PROPULSION SYSTEM', 'AIR BAGS',
    'STRUCTURE', 'VEHICLE SPEED CONTROL'
]

tiers        = [1, 2, 3]
tier_weights = [0.20, 0.50, 0.30]  # Tier 2 dominant — reflects real supply chains

n = 300
supplier_ids = [f'SUP{str(i).zfill(4)}' for i in range(1, n+1)]  # SUP0001, SUP0002 ...

df_suppliers = pd.DataFrame({
    'supplier_id'   : supplier_ids,
    'supplier_name' : [f"{random.choice(manufacturers)} - Unit {i}" for i in range(1, n+1)],
    'country'       : random.choices(countries, weights=country_weights, k=n),
    'part_category' : random.choices(part_categories, k=n),
    'tier'          : random.choices(tiers, weights=tier_weights, k=n)
})

print(df_suppliers.shape)
print(df_suppliers['country'].value_counts())
print(df_suppliers['tier'].value_counts())

(300, 5)
country
India          92
Japan          41
Germany        40
China          36
South Korea    29
USA            28
Mexico         19
Thailand       15
Name: count, dtype: int64
tier
2    148
3     96
1     56
Name: count, dtype: int64


In [9]:
# STEP 7: ASSIGN SUPPLIERS TO ORDERS
# Each order in DataCo gets assigned a supplier based on its part_category
# Suppliers only get assigned orders in their own part category

category_supplier_map = {}
for category in part_categories:
    cat_suppliers = df_suppliers[df_suppliers['part_category'] == category]['supplier_id'].tolist()
    category_supplier_map[category] = cat_suppliers

random.seed(42)
df_dataco['supplier_id'] = df_dataco['part_category'].apply(
    lambda cat: random.choice(category_supplier_map.get(cat, ['UNKNOWN']))
)

# Merge supplier details into orders
# We drop part_category from suppliers first because it already exists in df_dataco
df_suppliers_merge = df_suppliers.drop(columns=['part_category'])
df_orders = df_dataco.merge(df_suppliers_merge, on='supplier_id', how='left')

print(df_orders.shape)
df_orders[['order_id', 'supplier_id', 'supplier_name', 'country', 'tier',
           'part_category', 'delay_days', 'is_delayed', 'delivery_status']].head()

(160351, 24)


,order_id,supplier_id,supplier_name,country,tier,part_category,delay_days,is_delayed,delivery_status
0,28744,SUP0215,Minda Industries - Unit 215,Germany,2,ENGINE,3,True,Late delivery
1,45461,SUP0071,Gabriel India - Unit 71,Germany,2,STRUCTURE,0,False,Shipping on time
2,31115,SUP0004,Aptiv - Unit 4,South Korea,2,SERVICE BRAKES,4,True,Late delivery
3,45766,SUP0106,Aisin - Unit 106,India,2,STRUCTURE,0,False,Shipping on time
4,47752,SUP0111,Endurance Technologies - Unit 111,South Korea,2,SERVICE BRAKES,1,True,Late delivery


In [10]:
# STEP 8: SUPPLIER RISK SCORING 
# Score each supplier on 3 dimensions:
#   40% — Average delay days (delivery speed)
#   35% — Delay rate (% of orders that were late)
#   25% — Defect count from NHTSA (quality risk in their part category)
#
# MinMaxScaler normalises each metric to 0-100 so they can be combined
# Higher score = higher risk

# Aggregate per-supplier delivery metrics
supplier_stats = df_orders.groupby(
    ['supplier_id', 'supplier_name', 'country', 'tier', 'part_category']
).agg(
    total_orders    = ('order_id', 'count'),
    avg_delay       = ('delay_days', 'mean'),
    delay_rate      = ('is_delayed', 'mean'),      # proportion of orders delayed
    avg_order_value = ('order_value', 'mean'),
    total_profit    = ('profit', 'sum')
).reset_index()

# Get NHTSA defect count per component category
defect_counts = (
    df_nhtsa.groupby('component_category')['complaint_id']
    .count()
    .reset_index()
)
defect_counts.columns = ['part_category', 'defect_count']

# Join defect counts onto supplier stats
supplier_stats = supplier_stats.merge(defect_counts, on='part_category', how='left')
supplier_stats['defect_count'] = supplier_stats['defect_count'].fillna(0)

# Normalise each metric to 0-100 using MinMaxScaler
# MinMaxScaler formula: (value - min) / (max - min) * 100
scaler = MinMaxScaler(feature_range=(0, 100))

supplier_stats['delay_score']      = scaler.fit_transform(supplier_stats[['avg_delay']])
supplier_stats['delay_rate_score'] = scaler.fit_transform(supplier_stats[['delay_rate']])
supplier_stats['defect_score']     = scaler.fit_transform(supplier_stats[['defect_count']])

# Weighted final risk score
supplier_stats['risk_score'] = (
    0.40 * supplier_stats['delay_score'] +
    0.35 * supplier_stats['delay_rate_score'] +
    0.25 * supplier_stats['defect_score']
).round(1)

# Assign risk category label
def risk_label(score):
    if score >= 70:   return 'High Risk'
    elif score >= 40: return 'Medium Risk'
    else:             return 'Low Risk'

supplier_stats['risk_category'] = supplier_stats['risk_score'].apply(risk_label)

print(supplier_stats['risk_category'].value_counts())
print()
print("Top 10 Highest Risk Suppliers:")
supplier_stats.sort_values('risk_score', ascending=False)[[
    'supplier_name', 'country', 'tier', 'part_category',
    'avg_delay', 'delay_rate', 'risk_score', 'risk_category'
]].head(10)

risk_category
Medium Risk    226
Low Risk        45
High Risk       29
Name: count, dtype: int64

Top 10 Highest Risk Suppliers:


,supplier_name,country,tier,part_category,avg_delay,delay_rate,risk_score,risk_category
100,Uno Minda - Unit 101,China,2,AIR BAGS,0.75,0.61,79.60,High Risk
219,Tata AutoComp - Unit 220,USA,1,AIR BAGS,0.71,0.62,79.20,High Risk
246,Schaeffler India - Unit 247,India,1,ENGINE,0.66,0.59,78.10,High Risk
250,Tata AutoComp - Unit 251,Germany,2,ENGINE,0.65,0.59,76.70,High Risk
11,Aptiv - Unit 12,USA,2,ENGINE,0.63,0.59,76.70,High Risk
181,Uno Minda - Unit 182,Japan,3,ENGINE,0.62,0.59,75.40,High Risk
151,Sundram Fasteners - Unit 152,Mexico,3,ELECTRICAL SYSTEM,0.62,0.60,75.40,High Risk
234,Magna International - Unit 235,South Korea,2,ELECTRICAL SYSTEM,0.63,0.59,74.30,High Risk
213,ZF Friedrichshafen - Unit 214,Germany,2,ENGINE,0.62,0.58,74.20,High Risk
58,Tenneco - Unit 59,India,2,ENGINE,0.65,0.58,73.90,High Risk


In [11]:
# STEP 9: EXPORT CLEAN FILES FOR MYSQL 
# Fix date format first — MySQL needs YYYY-MM-DD HH:MM:SS format
df_orders['order_date'] = pd.to_datetime(df_orders['order_date']).dt.strftime('%Y-%m-%d %H:%M:%S')
df_orders['ship_date']  = pd.to_datetime(df_orders['ship_date']).dt.strftime('%Y-%m-%d %H:%M:%S')

# Table 1 — Supplier risk scores (300 rows)
supplier_stats.to_csv('supplier_risk_scores.csv', index=False)

# Table 2 — Cleaned orders with supplier info (160K rows)
df_orders[[
    'order_id', 'supplier_id', 'part_category', 'order_date', 'ship_date',
    'days_shipping_scheduled', 'days_shipping_real', 'delay_days', 'is_delayed',
    'delivery_status', 'quantity', 'order_value', 'profit',
    'shipping_mode', 'order_country', 'region', 'market'
]].to_csv('orders_clean.csv', index=False)

# Table 3 — NHTSA defects filtered to auto components only (746K rows)
auto_components = [
    'ENGINE', 'ELECTRICAL SYSTEM', 'SERVICE BRAKES', 'POWER TRAIN',
    'STEERING', 'SUSPENSION', 'FUEL/PROPULSION SYSTEM', 'AIR BAGS',
    'STRUCTURE', 'VEHICLE SPEED CONTROL'
]
df_nhtsa[df_nhtsa['component_category'].isin(auto_components)].to_csv('defects_clean.csv', index=False)

print("✅ All 3 files exported:")
print(f"   supplier_risk_scores.csv — {supplier_stats.shape[0]} rows")
print(f"   orders_clean.csv         — {df_orders.shape[0]} rows")
print(f"   defects_clean.csv        — {df_nhtsa[df_nhtsa['component_category'].isin(auto_components)].shape[0]} rows")

✅ All 3 files exported:
   supplier_risk_scores.csv — 300 rows
   orders_clean.csv         — 160351 rows
   defects_clean.csv        — 746257 rows
